# Практическая работа №3: обработка данных

Этот ноутбук производит весь расчет для отчета: предобработка анкеты, факторный анализ объектов и характеристик экспертов, критерий Фридмана, коэффициент конкордации, регрессия, графики, LaTeX-таблицы и презентация.

## 1. Импорт библиотек и настройки проекта

In [1]:
# Автоустановка зависимостей для VS Code / Colab / Jupyter
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "openpyxl": "openpyxl",
    "pptx": "python-pptx",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Устанавливаю недостающие пакеты:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Все зависимости уже установлены.")


Все зависимости уже установлены.


In [2]:
import os
import re
import math
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from pptx.dml.color import RGBColor

try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()

# Если запускаете notebook в Colab / VS Code из папки, где лежит lab-3.zip,
# архив распакуется автоматически.
ZIP_CANDIDATES = [ROOT / 'lab-3.zip', Path.cwd() / 'lab-3.zip', Path.cwd().parent / 'lab-3.zip']
ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if ZIP_PATH is not None and not (ZIP_PATH.parent / 'lab-3').exists() and not (ROOT / 'data').exists():
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(ZIP_PATH.parent)
    print(f'Архив распакован: {ZIP_PATH.name}')

# Поддерживаются варианты запуска:
# 1) notebook лежит в корне проекта рядом с data/
# 2) notebook лежит в src/ внутри проекта
# 3) notebook запущен из папки, где есть lab-3/data/
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists() and (ROOT / 'lab-3' / 'data').exists():
    ROOT = ROOT / 'lab-3'

DATA_PATH_CANDIDATES = [
    ROOT / 'data' / 'otvety.xlsx',
    ROOT / 'data' / 'otvety_ispravleno.xlsx',
    Path.cwd() / 'data' / 'otvety.xlsx',
    Path.cwd() / 'data' / 'otvety_ispravleno.xlsx',
    Path.cwd().parent / 'data' / 'otvety.xlsx',
    Path.cwd().parent / 'data' / 'otvety_ispravleno.xlsx',
    Path.cwd() / 'lab-3' / 'data' / 'otvety.xlsx',
    Path.cwd() / 'lab-3' / 'data' / 'otvety_ispravleno.xlsx',
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    searched = '\n'.join(str(p) for p in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(
        'Не найден файл с ответами. Положите lab-3.zip рядом с notebook или откройте notebook из корня проекта.\n'
        f'Проверенные пути:\n{searched}'
    )

ASSETS = ROOT / 'assets'
GEN = ROOT / 'generated'
TABLES = ROOT / 'tables'
for p in [ASSETS, GEN, TABLES]:
    p.mkdir(exist_ok=True)
print(f'ROOT = {ROOT}')
print(f'DATA_PATH = {DATA_PATH}')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 160
plt.rcParams['savefig.dpi'] = 220
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

SERVICES = ['Яндекс Еда', 'Самокат', 'Delivery Club', 'Купер', 'Ozon Fresh']
CRITERIA = ['цена', 'скорость доставки', 'общее качество']
SHORT_SERVICE = {
    'Яндекс Еда': 'Яндекс',
    'Самокат': 'Самокат',
    'Delivery Club': 'Delivery Club',
    'Купер': 'Купер',
    'Ozon Fresh': 'Ozon Fresh',
}
SHORT_CRIT = {
    'цена': 'ц.',
    'скорость доставки': 'скор.',
    'общее качество': 'кач.',
}
FREQ_MAP = {
    'Не пользуюсь': 0,
    'Реже 1 раза в месяц': 1,
    '1-2 раза в месяц': 2,
    '1 раз в неделю': 3,
    '2-3 раза в неделю': 4,
    'Почти каждый день': 5,
}
COMPARE_MAP = {'Никогда': 0, 'Редко': 1, 'Иногда': 2, 'Часто': 3, 'Всегда': 4}
USED_MAP = {'Нет': 0, 'Да': 1}


ROOT = /home/a/Desktop/labs-guap/2course/2term/SPoI/SPoI/lab-3
DATA_PATH = /home/a/Desktop/labs-guap/2course/2term/SPoI/SPoI/lab-3/data/otvety.xlsx


## 2. Вспомогательные функции для LaTeX и форматирования

In [3]:
def tex_escape(x):
    if pd.isna(x):
        return ''
    s = str(x)
    repl = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    return ''.join(repl.get(ch, ch) for ch in s)


def fmt_num(x, nd=3):
    if pd.isna(x):
        return ''
    if abs(x) >= 1000:
        s = f'{x:,.0f}' if nd == 0 else f'{x:,.{nd}f}'
        s = s.replace(',', ' ')
    else:
        s = f'{x:.{nd}f}'
    return s.replace('.', ',')


def fmt_p(p):
    if p < 0.001:
        return '< 0,001'
    return fmt_num(p, 3)


def write_simple_table(path, headers, rows, align=None):
    if align is None:
        align = ['l'] * len(headers)
    colspec = ''.join(align)
    lines = ['{%', r'\small', r'\setlength{\tabcolsep}{3.0pt}', r'\renewcommand{\arraystretch}{1.12}', r'\begin{adjustbox}{max width=\textwidth}', r'\begin{tabular}{' + colspec + '}', r'\toprule']
    lines.append(' & '.join(tex_escape(h) for h in headers) + r' \\')
    lines.append(r'\midrule')
    for row in rows:
        lines.append(' & '.join(tex_escape(v) for v in row) + r' \\')
    lines.extend([r'\bottomrule', r'\end{tabular}', r'\end{adjustbox}', '}'])
    Path(path).write_text('\n'.join(lines), encoding='utf-8')


def write_longtable(path, headers, rows, colspec, caption=None, label=None, fontsize=r'\small'):
    lines = []
    lines.append('{%')
    lines.append(fontsize)
    lines.append(r'\setlength{\tabcolsep}{3.2pt}')
    lines.append(r'\renewcommand{\arraystretch}{1.08}')
    lines.append(r'\begin{longtable}{' + colspec + '}')
    if caption:
        cap = r'\caption{' + tex_escape(caption) + '}'
        if label:
            cap += r'\label{' + label + '}'
        cap += r'\\'
        lines.append(cap)
    lines.append(r'\toprule')
    lines.append(' & '.join(tex_escape(h) for h in headers) + r' \\')
    lines.append(r'\midrule')
    lines.append(r'\endfirsthead')
    lines.append(r'\toprule')
    lines.append(' & '.join(tex_escape(h) for h in headers) + r' \\')
    lines.append(r'\midrule')
    lines.append(r'\endhead')
    lines.append(r'\midrule')
    lines.append(r'\multicolumn{' + str(len(headers)) + r'}{r}{Продолжение на следующей странице}\\')
    lines.append(r'\endfoot')
    lines.append(r'\bottomrule')
    lines.append(r'\endlastfoot')
    for row in rows:
        lines.append(' & '.join(tex_escape(v) for v in row) + r' \\')
    lines.append(r'\end{longtable}')
    lines.append('}')
    Path(path).write_text('\n'.join(lines), encoding='utf-8')


def varimax(Phi, gamma=1.0, q=100, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda ** 3 - (gamma / p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d / d_old < 1 + tol:
            break
    return Phi @ R, R


## 3. Загрузка и подготовка данных

In [4]:
def find_column(df, candidates, required_keywords=None):
    """Найти столбец, даже если в названии отличается валюта/пунктуация."""
    for col in candidates:
        if col in df.columns:
            return col
    if required_keywords:
        lowered = {col: str(col).lower() for col in df.columns}
        for col, low in lowered.items():
            if all(keyword.lower() in low for keyword in required_keywords):
                return col
    raise KeyError(
        'Не найден нужный столбец. Проверенные варианты: '
        + ', '.join(candidates)
    )


def prepare_data():
    df = pd.read_excel(DATA_PATH, sheet_name='Ответы_объединенные')
    global SPEND_COL
    SPEND_COL = find_column(
        df,
        [
            'Сколько примерно тратите на доставку еды в месяц, руб.',
            'Сколько примерно тратите на доставку еды в месяц, ₽',
            'Сколько примерно тратите на доставку еды в месяц, руб',
        ],
        required_keywords=['сколько примерно тратите', 'месяц'],
    )
    service_cols = {}
    for service in SERVICES:
        cols = [c for c in df.columns if c.startswith(service + ' —')]
        # normalize order: price, speed, quality even for Купер in source file
        ordered = []
        for criterion in CRITERIA:
            found = [c for c in cols if criterion in c]
            if not found:
                raise ValueError(f'Не найден столбец {service} — {criterion}')
            ordered.append(found[0])
        service_cols[service] = ordered
        df[f'{service}_интегральная оценка'] = df[ordered].mean(axis=1)
    df['Частота_числ'] = df['Как часто вы пользуетесь сервисами доставки еды?'].map(FREQ_MAP)
    df['Сравнение_цен_числ'] = df['Как часто сравниваете цены в разных сервисах перед заказом?'].map(COMPARE_MAP)
    df['Пользовались_3мес_числ'] = df['Пользовались ли сервисами доставки еды за последние 3 месяца?'].map(USED_MAP)
    df['Расходы_log'] = np.log1p(pd.to_numeric(df[SPEND_COL], errors='coerce').fillna(0))
    return df, service_cols


## 4. Расчет статистики, факторного анализа, согласованности и регрессии

In [5]:
def compute_results(df, service_cols):
    object_cols = [f'{s}_интегральная оценка' for s in SERVICES]
    n = len(df)
    k = len(SERVICES)

    service_summary_rows = []
    for service in SERVICES:
        row = {'Сервис': service}
        for criterion, col in zip(CRITERIA, service_cols[service]):
            row[criterion] = df[col].mean()
        scores = df[f'{service}_интегральная оценка']
        row['Интегральная оценка'] = scores.mean()
        row['Станд. отклонение'] = scores.std(ddof=1)
        row['Медиана'] = scores.median()
        service_summary_rows.append(row)
    service_summary = pd.DataFrame(service_summary_rows)

    ranks = np.apply_along_axis(lambda x: rankdata(x, method='average'), 1, df[object_cols].to_numpy(float))
    mean_ranks = pd.Series(ranks.mean(axis=0), index=SERVICES, name='Средний ранг')
    service_summary = service_summary.merge(mean_ranks.rename_axis('Сервис').reset_index(), on='Сервис')

    X_obj = df[object_cols].to_numpy(float)
    Z_obj = StandardScaler().fit_transform(X_obj)
    pca_obj = PCA().fit(Z_obj)
    n_obj_factors = 2
    load_obj = pca_obj.components_[:n_obj_factors].T * np.sqrt(pca_obj.explained_variance_[:n_obj_factors])
    rot_obj, R_obj = varimax(load_obj)
    obj_loadings = pd.DataFrame(rot_obj, index=SERVICES, columns=['F1', 'F2'])
    obj_loadings['h2'] = np.sum(rot_obj ** 2, axis=1)
    obj_eigen = pd.DataFrame({
        'Фактор': [f'F{i+1}' for i in range(len(pca_obj.explained_variance_))],
        'Собственное значение': pca_obj.explained_variance_,
        'Доля дисперсии': pca_obj.explained_variance_ratio_,
        'Накопленная доля': np.cumsum(pca_obj.explained_variance_ratio_),
    })

    friedman = stats.friedmanchisquare(*[df[col] for col in object_cols])
    W = friedman.statistic / (n * (k - 1))

    try:
        candidates = pd.read_excel(DATA_PATH, sheet_name='Кандидаты_на_удаление', header=None)
        bad_ids = [x for x in candidates.iloc[:, 0].dropna().astype(str).tolist() if re.match(r'R\d{3}', x)]
    except Exception:
        bad_ids = []
    df_clean = df[~df['ID респондента'].isin(bad_ids)].copy()
    ranks_clean = np.apply_along_axis(lambda x: rankdata(x, method='average'), 1, df_clean[object_cols].to_numpy(float))
    friedman_clean = stats.friedmanchisquare(*[df_clean[col] for col in object_cols])
    W_clean = friedman_clean.statistic / (len(df_clean) * (k - 1))
    mean_ranks_clean = pd.Series(ranks_clean.mean(axis=0), index=SERVICES, name='Средний ранг без кандидатов')

    expert_vars = {
        'важность цены': 'Насколько важна цена при выборе сервиса доставки?',
        'важность скорости': 'Насколько важна скорость доставки?',
        'важность скидок': 'Насколько важны скидки и промокоды?',
        'важность ассортимента': 'Насколько важен ассортимент ресторанов/магазинов?',
        'важность приложения': 'Насколько важна удобность приложения?',
        'важность поддержки': 'Насколько важна служба поддержки?',
        'точность времени': 'Насколько важна точность времени доставки?',
        'частота заказов': 'Частота_числ',
        'расходы (log)': 'Расходы_log',
        'готовность рекомендовать': 'Готовы рекомендовать чаще используемый сервис другим студентам?',
        'сравнение цен': 'Сравнение_цен_числ',
        'пользовался 3 мес.': 'Пользовались_3мес_числ',
    }
    expert_names = list(expert_vars.keys())
    E = df[list(expert_vars.values())].astype(float)
    Z_exp = StandardScaler().fit_transform(E)
    pca_exp = PCA().fit(Z_exp)
    n_exp_factors = 4
    load_exp = pca_exp.components_[:n_exp_factors].T * np.sqrt(pca_exp.explained_variance_[:n_exp_factors])
    rot_exp, R_exp = varimax(load_exp)
    exp_loadings = pd.DataFrame(rot_exp, index=expert_names, columns=[f'F{i+1}' for i in range(n_exp_factors)])
    exp_loadings['h2'] = np.sum(rot_exp ** 2, axis=1)
    exp_eigen = pd.DataFrame({
        'Фактор': [f'F{i+1}' for i in range(len(pca_exp.explained_variance_))],
        'Собственное значение': pca_exp.explained_variance_,
        'Доля дисперсии': pca_exp.explained_variance_ratio_,
        'Накопленная доля': np.cumsum(pca_exp.explained_variance_ratio_),
    })

    def fav_score(row):
        fav = row['Какой сервис доставки еды вы используете чаще всего?']
        if fav in SERVICES:
            return row[f'{fav}_интегральная оценка']
        return row[object_cols].mean()

    df['Оценка_частого_сервиса'] = df.apply(fav_score, axis=1)
    reg_features = [
        'Оценка_частого_сервиса',
        'Частота_числ',
        'Расходы_log',
        'Сравнение_цен_числ',
        'Насколько важна цена при выборе сервиса доставки?',
        'Насколько важна скорость доставки?',
        'Насколько важны скидки и промокоды?',
        'Насколько важна удобность приложения?',
        'Насколько важна служба поддержки?',
        'Насколько важна точность времени доставки?',
    ]
    reg_feature_labels = {
        'Оценка_частого_сервиса': 'оценка чаще используемого сервиса',
        'Частота_числ': 'частота заказов',
        'Расходы_log': 'расходы (log)',
        'Сравнение_цен_числ': 'сравнение цен',
        'Насколько важна цена при выборе сервиса доставки?': 'важность цены',
        'Насколько важна скорость доставки?': 'важность скорости',
        'Насколько важны скидки и промокоды?': 'важность скидок',
        'Насколько важна удобность приложения?': 'важность приложения',
        'Насколько важна служба поддержки?': 'важность поддержки',
        'Насколько важна точность времени доставки?': 'точность времени',
    }
    y = df['Готовы рекомендовать чаще используемый сервис другим студентам?'].astype(float)
    X = df[reg_features].astype(float)
    X_std = pd.DataFrame(StandardScaler().fit_transform(X), columns=[reg_feature_labels[c] for c in reg_features])
    model = sm.OLS(y, sm.add_constant(X_std)).fit()
    reg_table = pd.DataFrame({
        'Переменная': ['константа'] + X_std.columns.tolist(),
        'b': model.params.values,
        'SE': model.bse.values,
        't': model.tvalues.values,
        'p': model.pvalues.values,
    })

    return {
        'n': n,
        'k': k,
        'service_cols': service_cols,
        'object_cols': object_cols,
        'service_summary': service_summary,
        'mean_ranks': mean_ranks,
        'mean_ranks_clean': mean_ranks_clean,
        'obj_eigen': obj_eigen,
        'obj_loadings': obj_loadings,
        'obj_pca': pca_obj,
        'obj_rotation': R_obj,
        'exp_eigen': exp_eigen,
        'exp_loadings': exp_loadings,
        'friedman': friedman,
        'W': W,
        'bad_ids': bad_ids,
        'friedman_clean': friedman_clean,
        'W_clean': W_clean,
        'n_clean': len(df_clean),
        'df_clean': df_clean,
        'expert_vars': expert_vars,
        'model': model,
        'reg_table': reg_table,
    }


## 5. Экспорт CSV-таблиц

In [6]:
def generate_csv(df, results):
    object_cols = results['object_cols']
    df.to_csv(TABLES / 'cleaned_survey_with_scores.csv', index=False, encoding='utf-8-sig')
    df[['ID респондента'] + object_cols].to_csv(TABLES / 'expert_object_matrix_integral_scores.csv', index=False, encoding='utf-8-sig')
    results['service_summary'].to_csv(TABLES / 'service_summary.csv', index=False, encoding='utf-8-sig')
    results['obj_eigen'].to_csv(TABLES / 'object_factor_eigenvalues.csv', index=False, encoding='utf-8-sig')
    results['obj_loadings'].to_csv(TABLES / 'object_factor_loadings.csv', encoding='utf-8-sig')
    results['exp_eigen'].to_csv(TABLES / 'expert_factor_eigenvalues.csv', index=False, encoding='utf-8-sig')
    results['exp_loadings'].to_csv(TABLES / 'expert_factor_loadings.csv', encoding='utf-8-sig')
    results['reg_table'].to_csv(TABLES / 'regression_recommendation.csv', index=False, encoding='utf-8-sig')


## 6. Генерация LaTeX-таблиц для отчета

In [7]:
def generate_tex_tables(df, results):
    service_summary = results['service_summary'].copy()
    service_summary = service_summary.sort_values('Интегральная оценка', ascending=False)
    rows = []
    for _, r in service_summary.iterrows():
        rows.append([
            r['Сервис'], fmt_num(r['цена'], 2), fmt_num(r['скорость доставки'], 2),
            fmt_num(r['общее качество'], 2), fmt_num(r['Интегральная оценка'], 2),
            fmt_num(r['Станд. отклонение'], 2), fmt_num(r['Средний ранг'], 2),
        ])
    write_simple_table(GEN / 'table_service_summary.tex',
        ['Сервис', 'Цена', 'Скорость', 'Качество', 'Средняя оценка', 's', 'Средний ранг'],
        rows, ['l','r','r','r','r','r','r'])

    # demographics and survey description
    demo_rows = []
    for block, col in [('Пол', 'Пол'), ('Возраст', 'Возраст'), ('Группа', 'Учебная группа'),
                       ('Частота использования', 'Как часто вы пользуетесь сервисами доставки еды?'),
                       ('Чаще используемый сервис', 'Какой сервис доставки еды вы используете чаще всего?'),
                       ('Сравнение цен', 'Как часто сравниваете цены в разных сервисах перед заказом?')]:
        vc = df[col].value_counts().sort_index() if col in ['Возраст', 'Учебная группа'] else df[col].value_counts()
        for cat, cnt in vc.items():
            demo_rows.append([block, str(cat), str(int(cnt)), fmt_num(cnt / len(df) * 100, 1)])
    write_longtable(GEN / 'table_demographics.tex', ['Блок', 'Категория', 'n', '%'], demo_rows,
                    'L{4.1cm}L{5.2cm}R{1.1cm}R{1.3cm}',
                    'Распределение ответов по ключевым вопросам анкеты', 'tab:demographics', r'\small')

    # object factor tables
    eig_rows = []
    for _, r in results['obj_eigen'].head(5).iterrows():
        eig_rows.append([r['Фактор'], fmt_num(r['Собственное значение'], 3), fmt_num(r['Доля дисперсии']*100, 1), fmt_num(r['Накопленная доля']*100, 1)])
    write_simple_table(GEN / 'table_object_eigen.tex', ['Фактор', 'Собственное значение', 'Доля, %', 'Накопл., %'], eig_rows, ['l','r','r','r'])

    rows = []
    for service, r in results['obj_loadings'].iterrows():
        rows.append([service, fmt_num(r['F1'], 3), fmt_num(r['F2'], 3), fmt_num(r['h2'], 3)])
    write_simple_table(GEN / 'table_object_loadings.tex', ['Сервис', 'F1', 'F2', 'h²'], rows, ['l','r','r','r'])

    # expert factor tables
    exp_eig_rows = []
    for _, r in results['exp_eigen'].head(6).iterrows():
        exp_eig_rows.append([r['Фактор'], fmt_num(r['Собственное значение'], 3), fmt_num(r['Доля дисперсии']*100, 1), fmt_num(r['Накопленная доля']*100, 1)])
    write_simple_table(GEN / 'table_expert_eigen.tex', ['Фактор', 'Собственное значение', 'Доля, %', 'Накопл., %'], exp_eig_rows, ['l','r','r','r'])

    exp_rows = []
    for var, r in results['exp_loadings'].iterrows():
        exp_rows.append([var] + [fmt_num(r[f'F{i}'], 3) for i in range(1,5)] + [fmt_num(r['h2'], 3)])
    write_longtable(GEN / 'table_expert_loadings.tex', ['Переменная', 'F1', 'F2', 'F3', 'F4', 'h²'], exp_rows,
                    'L{5.2cm}R{1.35cm}R{1.35cm}R{1.35cm}R{1.35cm}R{1.2cm}',
                    'Факторные нагрузки характеристик экспертов после varimax-вращения', 'tab:expert_loadings', r'\small')

    # Friedman and W table
    fr = results['friedman']
    frc = results['friedman_clean']
    fr_rows = [
        ['До удаления кандидатов', str(results['n']), fmt_num(fr.statistic, 3), fmt_p(fr.pvalue), fmt_num(results['W'], 3)],
        ['После удаления кандидатов', str(results['n_clean']), fmt_num(frc.statistic, 3), fmt_p(frc.pvalue), fmt_num(results['W_clean'], 3)],
    ]
    write_simple_table(GEN / 'table_friedman.tex', ['Вариант расчёта', 'n', 'chi2_F', 'p-value', 'W'], fr_rows, ['l','r','r','r','r'])

    rank_rows = []
    for s in SERVICES:
        rank_rows.append([s, fmt_num(results['mean_ranks'][s], 3), fmt_num(results['mean_ranks_clean'][s], 3)])
    write_simple_table(GEN / 'table_mean_ranks.tex', ['Сервис', 'Средний ранг', 'Средний ранг без кандидатов'], rank_rows, ['l','r','r'])

    cand_rows = []
    if results['bad_ids']:
        for bid in results['bad_ids']:
            row = df.loc[df['ID респондента'] == bid].iloc[0]
            cand_rows.append([bid, row['Пол'], str(row['Возраст']), str(row['Учебная группа']),
                              row['Как часто вы пользуетесь сервисами доставки еды?'], row['Какой сервис доставки еды вы используете чаще всего?']])
    write_simple_table(GEN / 'table_candidates.tex', ['ID', 'Пол', 'Возраст', 'Группа', 'Частота', 'Чаще используемый сервис'], cand_rows, ['l','l','r','r','l','l'])

    # Regression table
    reg_rows = []
    for _, r in results['reg_table'].iterrows():
        reg_rows.append([r['Переменная'], fmt_num(r['b'], 3), fmt_num(r['SE'], 3), fmt_num(r['t'], 2), fmt_p(r['p'])])
    write_longtable(GEN / 'table_regression.tex', ['Переменная', 'b', 'SE', 't', 'p-value'], reg_rows,
                    'L{7.0cm}R{1.3cm}R{1.3cm}R{1.1cm}R{1.4cm}',
                    'Результаты регрессионной модели готовности рекомендовать сервис', 'tab:regression', r'\small')

    # Appendix: raw survey split into portrait-oriented blocks
    demo_profile_cols = ['ID респондента', 'Пол', 'Возраст', 'Учебная группа',
                         'Как часто вы пользуетесь сервисами доставки еды?',
                         'Какой сервис доставки еды вы используете чаще всего?']
    profile_headers = ['ID', 'Пол', 'Возраст', 'Группа', 'Частота использования', 'Чаще используемый сервис']
    profile_rows = []
    for _, r in df[demo_profile_cols].iterrows():
        profile_rows.append([r['ID респондента'], r['Пол'], str(int(r['Возраст'])), str(int(r['Учебная группа'])),
                             r['Как часто вы пользуетесь сервисами доставки еды?'],
                             r['Какой сервис доставки еды вы используете чаще всего?']])

    demo_behavior_cols = ['ID респондента', SPEND_COL,
                          'Пользовались ли сервисами доставки еды за последние 3 месяца?',
                          'Готовы рекомендовать чаще используемый сервис другим студентам?',
                          'Как часто сравниваете цены в разных сервисах перед заказом?']
    behavior_headers = ['ID', 'Расходы, руб.', 'Использовал за 3 мес.', 'Рекомендация', 'Сравнение цен']
    behavior_rows = []
    for _, r in df[demo_behavior_cols].iterrows():
        behavior_rows.append([r['ID респондента'], fmt_num(r[SPEND_COL], 0),
                              r['Пользовались ли сервисами доставки еды за последние 3 месяца?'],
                              str(int(r['Готовы рекомендовать чаще используемый сервис другим студентам?'])),
                              r['Как часто сравниваете цены в разных сервисах перед заказом?']])

    demo_tex = []
    demo_tex.append(r'\subsection*{А.1 Профиль респондентов}')
    write_longtable(GEN / 'appendix_demographics_profile.tmp', profile_headers, profile_rows,
                    'L{1.05cm}L{2.2cm}R{1.35cm}R{1.35cm}L{4.2cm}L{4.0cm}',
                    'Профиль респондентов', 'tab:appendix_demo_profile', r'\footnotesize')
    demo_tex.append((GEN / 'appendix_demographics_profile.tmp').read_text(encoding='utf-8'))
    demo_tex.append(r'\subsection*{А.2 Итоговые поведенческие ответы}')
    write_longtable(GEN / 'appendix_demographics_behavior.tmp', behavior_headers, behavior_rows,
                    'L{1.05cm}R{2.2cm}C{2.65cm}C{2.3cm}L{4.0cm}',
                    'Итоговые поведенческие ответы респондентов', 'tab:appendix_demo_behavior', r'\footnotesize')
    demo_tex.append((GEN / 'appendix_demographics_behavior.tmp').read_text(encoding='utf-8'))
    (GEN / 'appendix_demographics_full.tex').write_text('\n\n'.join(demo_tex), encoding='utf-8')
    for tmp in ['appendix_demographics_profile.tmp', 'appendix_demographics_behavior.tmp']:
        try:
            (GEN / tmp).unlink()
        except FileNotFoundError:
            pass

    obj_parts = [r'\subsection*{А.3 Матрица экспертных оценок объектов}']
    for service in SERVICES:
        cols = results['service_cols'][service]
        headers = ['ID', 'Цена', 'Скорость', 'Качество', 'Интегральная оценка']
        rows = []
        for _, r in df.iterrows():
            rows.append([r['ID респондента'], str(int(r[cols[0]])), str(int(r[cols[1]])), str(int(r[cols[2]])),
                         fmt_num(r[f'{service}_интегральная оценка'], 2)])
        path = GEN / f'appendix_object_{service.lower().replace(" ", "_").replace("/", "_")}.tmp'
        write_longtable(path, headers, rows,
                        'L{1.05cm}R{2.1cm}R{2.1cm}R{2.1cm}R{3.0cm}',
                        f'Оценки сервиса {service}', None, r'\footnotesize')
        obj_parts.append(path.read_text(encoding='utf-8'))
        path.unlink()
    (GEN / 'appendix_object_ratings_full.tex').write_text('\n\n'.join(obj_parts), encoding='utf-8')

    expert_cols = [
        'Насколько важна цена при выборе сервиса доставки?', 'Насколько важна скорость доставки?',
        'Насколько важны скидки и промокоды?', 'Насколько важен ассортимент ресторанов/магазинов?',
        'Насколько важна удобность приложения?', 'Насколько важна служба поддержки?',
        'Насколько важна точность времени доставки?'
    ]
    expert_headers = ['ID','Цена','Скорость','Скидки','Ассорт.','Прил.','Поддерж.','Точн.']
    expert_rows = []
    for _, r in df.iterrows():
        expert_rows.append([r['ID респондента']] + [str(int(r[c])) for c in expert_cols])
    expert_tex = [r'\subsection*{А.4 Характеристики экспертов}']
    write_longtable(GEN / 'appendix_expert_characteristics.tmp', expert_headers, expert_rows,
                    'L{1.05cm}' + 'R{1.42cm}'*7,
                    'Полные ответы по характеристикам экспертов', 'tab:appendix_experts', r'\scriptsize')
    expert_tex.append((GEN / 'appendix_expert_characteristics.tmp').read_text(encoding='utf-8'))
    (GEN / 'appendix_expert_characteristics_full.tex').write_text('\n\n'.join(expert_tex), encoding='utf-8')
    (GEN / 'appendix_expert_characteristics.tmp').unlink()

    # Metrics as LaTeX macros
    model = results['model']
    metrics = {
        'NRespondents': results['n'],
        'NObjects': results['k'],
        'NGroups': df['Учебная группа'].nunique(),
        'FemaleCount': int((df['Пол'] == 'Женский').sum()),
        'MaleCount': int((df['Пол'] == 'Мужской').sum()),
        'UsedThreeMonthsCount': int((df['Пользовались ли сервисами доставки еды за последние 3 месяца?'] == 'Да').sum()),
        'MeanSpend': df[SPEND_COL].mean(),
        'MedianSpend': df[SPEND_COL].median(),
        'FriedmanChi': results['friedman'].statistic,
        'FriedmanP': results['friedman'].pvalue,
        'ConcordanceW': results['W'],
        'FriedmanCleanChi': results['friedman_clean'].statistic,
        'FriedmanCleanP': results['friedman_clean'].pvalue,
        'ConcordanceWClean': results['W_clean'],
        'ObjectFactorShare': results['obj_eigen']['Доля дисперсии'].head(2).sum() * 100,
        'ExpertFactorShare': results['exp_eigen']['Доля дисперсии'].head(4).sum() * 100,
        'RegRSq': model.rsquared,
        'RegAdjRSq': model.rsquared_adj,
        'RegF': model.fvalue,
        'RegFP': model.f_pvalue,
    }
    lines = []
    for name, val in metrics.items():
        if isinstance(val, (int, np.integer)):
            out = str(val)
        elif name.endswith('P') or name in ['FriedmanP','FriedmanCleanP','RegFP']:
            out = fmt_p(float(val))
        elif 'Share' in name:
            out = fmt_num(float(val), 1)
        elif 'Spend' in name:
            out = fmt_num(float(val), 0)
        else:
            out = fmt_num(float(val), 3)
        lines.append('\\newcommand{\\' + name + '}{' + out + '}')
    (GEN / 'metrics.tex').write_text('\n'.join(lines), encoding='utf-8')


## 7. Построение графиков

In [8]:
def generate_figures(df, results):
    # Favorite service bar
    vc = df['Какой сервис доставки еды вы используете чаще всего?'].value_counts()
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    bars = ax.bar(vc.index.astype(str), vc.values)
    ax.set_title('Какой сервис используется чаще всего')
    ax.set_ylabel('Количество респондентов')
    ax.set_xlabel('Сервис')
    ax.tick_params(axis='x', rotation=25)
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, str(int(b.get_height())), ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    fig.savefig(ASSETS / 'favorite_service_bar.png')
    plt.close(fig)

    # Frequency and spending histogram
    freq_order = list(FREQ_MAP.keys())
    counts = df['Как часто вы пользуетесь сервисами доставки еды?'].value_counts().reindex(freq_order).fillna(0)
    fig, ax = plt.subplots(figsize=(8.4, 4.6))
    ax.bar(counts.index, counts.values)
    ax.set_title('Частота использования доставки еды')
    ax.set_ylabel('Количество респондентов')
    ax.set_xlabel('Частота')
    ax.tick_params(axis='x', rotation=30)
    for i, val in enumerate(counts.values):
        ax.text(i, val + 0.2, str(int(val)), ha='center', fontsize=9)
    fig.tight_layout()
    fig.savefig(ASSETS / 'usage_frequency_bar.png')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.8, 4.5))
    ax.hist(df[SPEND_COL], bins=10, edgecolor='white')
    ax.axvline(df[SPEND_COL].median(), linestyle='--', linewidth=1.5, label='медиана')
    ax.set_title('Распределение месячных расходов на доставку')
    ax.set_xlabel('Расходы, руб.')
    ax.set_ylabel('Количество респондентов')
    ax.legend()
    fig.tight_layout()
    fig.savefig(ASSETS / 'monthly_spend_hist.png')
    plt.close(fig)

    # Grouped criteria means
    criteria_means = pd.DataFrame(index=SERVICES, columns=CRITERIA, dtype=float)
    for service in SERVICES:
        for criterion, col in zip(CRITERIA, results['service_cols'][service]):
            criteria_means.loc[service, criterion] = df[col].mean()
    fig, ax = plt.subplots(figsize=(8.8, 4.7))
    x = np.arange(len(SERVICES))
    width = 0.25
    for j, criterion in enumerate(CRITERIA):
        ax.bar(x + (j-1)*width, criteria_means[criterion].values, width, label=criterion)
    ax.set_ylim(0, 10)
    ax.set_xticks(x)
    ax.set_xticklabels(SERVICES, rotation=18)
    ax.set_ylabel('Средняя оценка по шкале 1-10')
    ax.set_title('Средние оценки сервисов по критериям')
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.18))
    fig.tight_layout()
    fig.savefig(ASSETS / 'service_criteria_grouped_bar.png')
    plt.close(fig)

    # Heatmap
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    im = ax.imshow(criteria_means.values.astype(float), aspect='auto')
    ax.set_xticks(np.arange(len(CRITERIA)))
    ax.set_xticklabels(['Цена', 'Скорость', 'Качество'])
    ax.set_yticks(np.arange(len(SERVICES)))
    ax.set_yticklabels(SERVICES)
    ax.set_title('Тепловая карта средних оценок')
    for i in range(len(SERVICES)):
        for j in range(len(CRITERIA)):
            ax.text(j, i, fmt_num(criteria_means.iloc[i, j], 2), ha='center', va='center', fontsize=10)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Средняя оценка')
    fig.tight_layout()
    fig.savefig(ASSETS / 'service_criteria_heatmap.png')
    plt.close(fig)

    # Object factor map
    load = results['obj_loadings'][['F1','F2']]
    fig, ax = plt.subplots(figsize=(6.6, 5.4))
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    for service, row in load.iterrows():
        ax.scatter(row['F1'], row['F2'], s=80)
        ax.text(row['F1'] + 0.02, row['F2'] + 0.02, service, fontsize=10)
    ax.set_xlabel('F1')
    ax.set_ylabel('F2')
    ax.set_title('Карта факторных нагрузок для оцениваемых объектов')
    ax.set_xlim(load['F1'].min() - 0.25, load['F1'].max() + 0.35)
    ax.set_ylim(load['F2'].min() - 0.25, load['F2'].max() + 0.35)
    fig.tight_layout()
    fig.savefig(ASSETS / 'object_factor_map.png')
    plt.close(fig)

    # Mean ranks bar
    ranks = results['mean_ranks'].sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7.4, 4.3))
    ax.barh(ranks.index, ranks.values)
    ax.set_xlabel('Средний ранг (чем выше, тем лучше)')
    ax.set_title('Средние ранги сервисов по экспертным оценкам')
    for i, v in enumerate(ranks.values):
        ax.text(v + 0.03, i, fmt_num(v, 2), va='center')
    fig.tight_layout()
    fig.savefig(ASSETS / 'mean_ranks_bar.png')
    plt.close(fig)

    # Expert loadings heatmap
    exp = results['exp_loadings'][['F1','F2','F3','F4']]
    fig, ax = plt.subplots(figsize=(7.8, 6.0))
    im = ax.imshow(exp.values, aspect='auto', vmin=-1, vmax=1)
    ax.set_xticks(np.arange(4))
    ax.set_xticklabels(['F1', 'F2', 'F3', 'F4'])
    ax.set_yticks(np.arange(len(exp.index)))
    ax.set_yticklabels(exp.index)
    ax.set_title('Факторные нагрузки характеристик экспертов')
    for i in range(exp.shape[0]):
        for j in range(exp.shape[1]):
            val = exp.iloc[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('Нагрузка')
    fig.tight_layout()
    fig.savefig(ASSETS / 'expert_factor_loadings_heatmap.png')
    plt.close(fig)

    # Regression observed vs fitted
    model = results['model']
    fig, ax = plt.subplots(figsize=(5.8, 5.0))
    ax.scatter(df['Готовы рекомендовать чаще используемый сервис другим студентам?'], model.fittedvalues, alpha=0.8)
    mn, mx = 1, 10
    ax.plot([mn, mx], [mn, mx], linestyle='--', linewidth=1.0)
    ax.set_xlim(mn, mx)
    ax.set_ylim(mn, mx)
    ax.set_xlabel('Фактическая готовность рекомендовать')
    ax.set_ylabel('Прогноз модели')
    ax.set_title('Регрессионная модель: факт и прогноз')
    fig.tight_layout()
    fig.savefig(ASSETS / 'regression_actual_vs_predicted.png')
    plt.close(fig)


## 8. Функции для оформления презентации

In [9]:
def add_textbox(slide, x, y, w, h, text, font_size=20, bold=False, color=(20, 30, 40), align=PP_ALIGN.LEFT):
    box = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = box.text_frame
    tf.clear()
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(font_size)
    run.font.bold = bold
    run.font.color.rgb = RGBColor(*color)
    return box


def add_bullets(slide, x, y, w, h, items, font_size=18):
    box = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = box.text_frame
    tf.clear()
    for idx, item in enumerate(items):
        p = tf.paragraphs[0] if idx == 0 else tf.add_paragraph()
        p.text = item
        p.level = 0
        p.font.size = Pt(font_size)
        p.space_after = Pt(6)
    return box


def add_title(slide, title, subtitle=None):
    add_textbox(slide, 0.55, 0.25, 12.0, 0.45, title, 26, True)
    if subtitle:
        add_textbox(slide, 0.58, 0.73, 11.2, 0.35, subtitle, 13, False, (70, 80, 90))
    line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.55), Inches(1.15), Inches(12.2), Inches(0.03))
    line.fill.solid()
    line.fill.fore_color.rgb = RGBColor(70, 120, 160)
    line.line.color.rgb = RGBColor(70, 120, 160)


## 9. Генерация презентации

In [10]:
def generate_presentation(df, results):
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank = prs.slide_layouts[6]

    # Slide 1
    slide = prs.slides.add_slide(blank)
    add_textbox(slide, 0.9, 1.75, 11.6, 0.8, 'Факторный анализ и экспертная оценка сервисов доставки еды', 30, True, align=PP_ALIGN.CENTER)
    add_textbox(slide, 1.5, 2.75, 10.4, 0.6, 'Практическое задание №3: опрос студентов и обработка результатов', 18, False, (60, 70, 80), align=PP_ALIGN.CENTER)
    add_textbox(slide, 2.2, 5.55, 9.0, 0.45, f'Выборка: {results["n"]} респондентов, группы 4414–4417, 5 оцениваемых сервисов', 16, False, (60, 70, 80), align=PP_ALIGN.CENTER)

    # Slide 2
    slide = prs.slides.add_slide(blank)
    add_title(slide, '1. Анкета и выборка')
    add_bullets(slide, 0.8, 1.45, 5.4, 4.9, [
        'Онлайн письменный опрос среди студентов учебного потока 4414–4417.',
        '60 заполненных анкет: требование «не менее 30 человек» выполнено с запасом.',
        'Оценки сервисов даны по шкале 1–10: цена, скорость доставки, общее качество.',
        'Отдельный блок описывает характеристики экспертов: частота заказов, расходы, важность факторов выбора.'
    ], 17)
    slide.shapes.add_picture(str(ASSETS / 'usage_frequency_bar.png'), Inches(6.5), Inches(1.4), width=Inches(6.2))

    # Slide 3
    slide = prs.slides.add_slide(blank)
    add_title(slide, '2. Наглядное представление оценок')
    slide.shapes.add_picture(str(ASSETS / 'service_criteria_grouped_bar.png'), Inches(0.6), Inches(1.35), width=Inches(6.25))
    slide.shapes.add_picture(str(ASSETS / 'service_criteria_heatmap.png'), Inches(7.05), Inches(1.35), width=Inches(5.6))
    add_textbox(slide, 0.8, 6.55, 11.8, 0.35, 'В среднем выше всего оценивается Самокат, но структура оценок неоднородна: скорость и цена работают как разные источники предпочтения.', 14, False, (50, 60, 70))

    # Slide 4
    slide = prs.slides.add_slide(blank)
    add_title(slide, '3. Факторный анализ объектов')
    slide.shapes.add_picture(str(ASSETS / 'object_factor_map.png'), Inches(0.8), Inches(1.35), width=Inches(5.55))
    add_bullets(slide, 6.65, 1.5, 5.9, 4.5, [
        f'Два фактора по правилу собственных значений > 1 объясняют {fmt_num(results["obj_eigen"]["Доля дисперсии"].head(2).sum()*100, 1)}% дисперсии.',
        'F1 связан с положительным восприятием Самоката и Ozon Fresh.',
        'F2 отделяет Купер и Delivery Club от Яндекс Еды; это похоже на различие по типу ожиданий от сервиса.',
        'Факторные нагрузки показывают, что оценки сервисов не сводятся к одному простому рейтингу.'
    ], 17)

    # Slide 5
    slide = prs.slides.add_slide(blank)
    add_title(slide, '4. Факторный анализ характеристик экспертов')
    slide.shapes.add_picture(str(ASSETS / 'expert_factor_loadings_heatmap.png'), Inches(0.65), Inches(1.25), width=Inches(6.6))
    add_bullets(slide, 7.55, 1.45, 5.1, 4.7, [
        'F1: вовлеченность в доставку — частота заказов, расходы, рекомендация, опыт за последние 3 месяца.',
        'F2: ожидания к скорости и ассортименту.',
        'F3: ценовая и интерфейсная требовательность.',
        'F4: точность времени против роли поддержки.'
    ], 17)

    # Slide 6
    slide = prs.slides.add_slide(blank)
    add_title(slide, '5. Согласованность экспертов')
    slide.shapes.add_picture(str(ASSETS / 'mean_ranks_bar.png'), Inches(0.75), Inches(1.35), width=Inches(6.0))
    add_bullets(slide, 7.1, 1.6, 5.4, 4.2, [
        f'Критерий Фридмана: χ² = {fmt_num(results["friedman"].statistic, 2)}, p = {fmt_p(results["friedman"].pvalue)}.',
        f'Коэффициент конкордации: W = {fmt_num(results["W"], 3)}.',
        'Различия между сервисами статистически значимы, но сила согласованности невысокая.',
        'Удаление 3 формально сомнительных анкет почти не меняет результат.'
    ], 17)

    # Slide 7
    slide = prs.slides.add_slide(blank)
    add_title(slide, '6. Регрессионный анализ')
    slide.shapes.add_picture(str(ASSETS / 'regression_actual_vs_predicted.png'), Inches(0.9), Inches(1.35), width=Inches(5.6))
    add_bullets(slide, 6.9, 1.5, 5.7, 4.5, [
        f'Цель модели — объяснить готовность рекомендовать чаще используемый сервис.',
        f'R² = {fmt_num(results["model"].rsquared, 3)}, скорректированный R² = {fmt_num(results["model"].rsquared_adj, 3)}.',
        'Наиболее заметные предикторы: месячные расходы, сравнение цен, важность цены.',
        'Чем активнее студент пользуется доставкой, тем устойчивее выражена рекомендация.'
    ], 17)

    # Slide 8
    slide = prs.slides.add_slide(blank)
    add_title(slide, '7. Итоговые выводы')
    add_bullets(slide, 0.9, 1.45, 11.4, 4.8, [
        'Анкета позволяет одновременно оценить объекты, характеристики экспертов и согласованность экспертной группы.',
        'Самокат получает максимальный средний ранг; Яндекс Еда занимает вторую позицию, а Купер — наиболее слабую по совокупной оценке.',
        'Факторный анализ показывает, что восприятие сервисов многомерно: важны не только «средние оценки», но и структура предпочтений.',
        'Согласованность экспертов статистически подтверждена, однако W невысок: в теме доставки заметна индивидуальная модель потребления.',
        'Регрессия показывает, что рекомендация связана скорее с реальной вовлеченностью и расходами, чем с одной субъективной оценкой сервиса.'
    ], 19)

    prs.save(ROOT / 'presentation_task3.pptx')


## 10. Полный запуск пайплайна

In [ ]:
def main():
    df, service_cols = prepare_data()
    results = compute_results(df, service_cols)
    generate_csv(df, results)
    generate_tex_tables(df, results)
    generate_figures(df, results)
    generate_presentation(df, results)
    print(json.dumps({
        'n': results['n'],
        'friedman_chi2': results['friedman'].statistic,
        'friedman_p': results['friedman'].pvalue,
        'kendall_w': results['W'],
        'reg_r2': results['model'].rsquared,
        'assets': sorted([p.name for p in ASSETS.glob('*.png')])
    }, ensure_ascii=False, indent=2))


## 11. Запуск анализа

Выполните эту ячейку, чтобы пересоздать все таблицы, графики и презентацию.

In [ ]:
main()

/home/a/miniconda/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/home/a/miniconda/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


{
  "n": 60,
  "friedman_chi2": 33.10736468500459,
  "friedman_p": 1.1355214394493841e-06,
  "kendall_w": 0.1379473528541858,
  "reg_r2": 0.6364781072471168,
  "assets": [
    "expert_factor_loadings_heatmap.png",
    "favorite_service_bar.png",
    "mean_ranks_bar.png",
    "monthly_spend_hist.png",
    "object_factor_map.png",
    "regression_actual_vs_predicted.png",
    "service_criteria_grouped_bar.png",
    "service_criteria_heatmap.png",
    "usage_frequency_bar.png"
  ]
}
